# Generar rostros anime con el modelo de difusión (anime64)

Carga `best.safetensors` (U-Net de 88.2 M de parámetros, pesos EMA, paso 87 274) y genera imágenes de 64×64 desde ruido con DDIM.
No necesita el repositorio: la configuración de la red viene dentro del archivo de pesos.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*. Con GPU, 50 pasos DDIM por imagen suelen tardar pocos segundos.

El archivo de pesos está en `runs/anime64_4070ti/best.safetensors` de la entrega (353 MB). No lo publiques sin revisar la licencia del dataset (splcher/animefacedataset v3).

## 1. Dependencias y GPU

In [ ]:
!pip -q install diffusers==0.42.1 safetensors
import torch
print("torch", torch.__version__)
assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU"
print(torch.cuda.get_device_name(0))

## 2. Pesos
Elige **una** forma. `drive` es lo más cómodo si vas a reutilizar el notebook (sube el archivo una vez a tu Drive).
`upload` lo sube directo a la sesión de Colab y se pierde al cerrarla; son 353 MB.

In [ ]:
FUENTE = "upload"                                   # "upload" o "drive"
RUTA_DRIVE = "/content/drive/MyDrive/best.safetensors"  # solo si FUENTE = "drive"

if FUENTE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PESOS = RUTA_DRIVE
else:
    from google.colab import files
    subido = files.upload()                         # elige best.safetensors
    PESOS = next(iter(subido))
print("pesos:", PESOS)

## 3. Modelo y muestreador (misma lógica que `src/pc3/inference.py`)

In [ ]:
import json, time, math
import numpy as np
from PIL import Image
from safetensors import safe_open
from safetensors.torch import load_file
from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler

def cargar_modelo(ruta):
    with safe_open(ruta, framework="pt", device="cpu") as f:
        meta = json.loads(f.metadata()["pc3"])      # config de la U-Net + procedencia del entrenamiento
    modelo = UNet2DModel.from_config(meta["model_config"])
    modelo.load_state_dict(load_file(ruta, device="cpu"))
    return modelo.to("cuda").eval(), meta

def programa(pasos):
    base = DDPMScheduler(num_train_timesteps=1000, beta_start=0.0001, beta_end=0.02, beta_schedule="linear",
                         variance_type="fixed_small", prediction_type="epsilon", clip_sample=True)
    s = DDIMScheduler.from_config(base.config)      # DDIM determinista (eta = 0)
    s.set_timesteps(pasos, device="cuda")
    return s

def a_pil(t):
    a = ((t.detach().float().cpu().clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)
    return Image.fromarray(a.permute(1, 2, 0).numpy())

@torch.inference_mode()
def generar(modelo, semillas, pasos=50, fp16=True, capturar=False):
    s = programa(pasos)
    # Mismo ruido inicial que el repositorio: una semilla = una cara, aunque cambie el lote.
    x = torch.stack([torch.randn((3, 64, 64), generator=torch.Generator().manual_seed(k)) for k in semillas]).cuda()
    marcas = set(np.linspace(0, len(s.timesteps) - 1, min(12, len(s.timesteps)), dtype=int).tolist())
    cuadros = [a_pil(x[0])] if capturar else []
    for i, t in enumerate(s.timesteps):
        with torch.autocast("cuda", dtype=torch.float16, enabled=fp16):
            eps = modelo(x, t).sample
        x = s.step(eps.float(), t, x, eta=0.0).prev_sample
        if capturar and i in marcas:
            cuadros.append(a_pil(x[0]))
    return x.cpu(), cuadros

modelo, meta = cargar_modelo(PESOS)
print("parámetros:", f"{sum(p.numel() for p in modelo.parameters()):,}", "| paso de entrenamiento:", meta.get("step"), "| pesos:", meta.get("weights"))

## 4. Generar
`SEMILLA_INICIAL` fija las caras (la misma semilla da la misma imagen). Para caras distintas, cámbiala.
Con `PASOS = 50` es el protocolo con el que se midió FID 10.10; menos pasos es más rápido pero no está evaluado.

In [ ]:
N = 16
SEMILLA_INICIAL = 12345
PASOS = 50

generar(modelo, [0], pasos=1)                       # calentamiento de CUDA, fuera de la medición
torch.cuda.synchronize(); t0 = time.perf_counter()
imgs, _ = generar(modelo, list(range(SEMILLA_INICIAL, SEMILLA_INICIAL + N)), pasos=PASOS)
torch.cuda.synchronize(); seg = time.perf_counter() - t0
print(f"{N} imágenes en {seg:.1f} s  ({seg / N:.2f} s por imagen, lote de {N}, {PASOS} pasos DDIM)")

pil = [a_pil(t) for t in imgs]
cols = 4
grid = Image.new("RGB", (cols * 64, math.ceil(N / cols) * 64))
for i, im in enumerate(pil):
    grid.paste(im, ((i % cols) * 64, (i // cols) * 64))
display(grid.resize((grid.width * 4, grid.height * 4), Image.NEAREST))

## 5. Una sola imagen, con el proceso de eliminación de ruido

In [ ]:
t0 = time.perf_counter()
una, cuadros = generar(modelo, [SEMILLA_INICIAL], pasos=PASOS, capturar=True)
torch.cuda.synchronize(); print(f"1 imagen en {time.perf_counter() - t0:.2f} s")

tira = Image.new("RGB", (64 * len(cuadros), 64))
for i, c in enumerate(cuadros):
    tira.paste(c, (i * 64, 0))
display(tira.resize((tira.width * 2, tira.height * 2), Image.NEAREST))   # de ruido (izq.) a cara (der.)
cuadros[0].save("denoising.gif", save_all=True, append_images=cuadros[1:], duration=150, loop=0)

## 6. Guardar y descargar

In [ ]:
import os, zipfile
os.makedirs("imagenes", exist_ok=True)
for i, im in enumerate(pil):
    im.resize((256, 256), Image.NEAREST).save(f"imagenes/{SEMILLA_INICIAL + i:06}.png")   # 256x256 para verlas mejor; quita resize para 64x64 originales
grid.save("grid.png")
with zipfile.ZipFile("imagenes.zip", "w") as z:
    for f in sorted(os.listdir("imagenes")): z.write(f"imagenes/{f}")
    z.write("grid.png"); z.write("denoising.gif")

from google.colab import files
files.download("imagenes.zip")

## 7. SDEdit: de una foto al ruido y del ruido a una imagen nueva
Sube una foto (se recorta al centro y se reduce a 64×64). El modelo nunca vio fotos reales y no conserva la identidad: es una exploración cualitativa.
Una fuerza baja (0.3) conserva la foto; una alta (0.9) se acerca a una cara anime libre.

In [ ]:
from google.colab import files
subida = files.upload()                              # elige una foto (jpg/png)
FOTO = next(iter(subida))
FUERZAS = [0.5, 0.8]

from PIL import ImageOps
foto = ImageOps.fit(ImageOps.exif_transpose(Image.open(FOTO)).convert("RGB"), (64, 64), method=Image.Resampling.LANCZOS)
x0 = (torch.from_numpy(np.array(foto, dtype=np.float32).transpose(2, 0, 1)) / 127.5 - 1.0).unsqueeze(0).cuda()

@torch.inference_mode()
def sdedit(modelo, x0, fuerza, pasos=50, semilla=12345, fp16=True):
    s = programa(pasos)
    primero = pasos - max(1, min(pasos, round(pasos * fuerza)))
    ts = s.timesteps[primero:]
    ruido = torch.randn(x0.shape, generator=torch.Generator().manual_seed(semilla)).cuda()
    # proceso directo: la misma foto con cada vez más ruido (mismo ruido en todos los cuadros)
    marcas = torch.linspace(0, int(ts[0]), 6).round().long()[1:-1]
    hacia_ruido = [a_pil(s.add_noise(x0, ruido, t.reshape(1).cuda())[0]) for t in marcas]
    x = s.add_noise(x0, ruido, ts[:1])
    idx = set(np.linspace(0, len(ts) - 1, min(12, len(ts)), dtype=int).tolist())
    limpieza = [a_pil(x[0])]
    for i, t in enumerate(ts):
        with torch.autocast("cuda", dtype=torch.float16, enabled=fp16):
            eps = modelo(x, t).sample
        x = s.step(eps.float(), t, x, eta=0.0).prev_sample
        if i in idx:
            limpieza.append(a_pil(x[0]))
    return x.cpu(), hacia_ruido + limpieza

for f in FUERZAS:
    t0 = time.perf_counter()
    final, cuadros = sdedit(modelo, x0, f)
    secuencia = [foto] + cuadros
    tira = Image.new("RGB", (64 * len(secuencia), 64))
    for i, c in enumerate(secuencia):
        tira.paste(c, (i * 64, 0))
    print(f"fuerza {f}: {time.perf_counter() - t0:.1f} s   (original → ruido → imagen nueva)")
    display(tira.resize((tira.width * 2, tira.height * 2), Image.NEAREST))
    secuencia[0].save(f"sdedit_{f:.2f}.gif", save_all=True, append_images=secuencia[1:], duration=200, loop=0)
    tira.save(f"sdedit_{f:.2f}_proceso.png")
    a_pil(final[0]).save(f"sdedit_{f:.2f}.png")
files.download("sdedit_0.50.gif")                    # los demás archivos quedan en el panel de archivos de Colab